In [1]:
# ==========================================================
# Notebook 09: End-to-End PDF Chatbot
# ==========================================================

import os
import faiss
import pdfplumber
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_text_splitters import RecursiveCharacterTextSplitter

import torch

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
# -----------------------------
# Configuration
# -----------------------------

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

LLM_MODEL = "microsoft/Phi-3-mini-4k-instruct"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 3
MAX_NEW_TOKENS = 256

In [3]:
class PDFParser:

    def parse(self, pdf_path):

        pages = []

        with pdfplumber.open(pdf_path) as pdf:

            for page_no, page in enumerate(pdf.pages, start=1):

                text = page.extract_text()

                if text:

                    pages.append({"page_number": page_no, "text": text})

        return pd.DataFrame(pages)

In [4]:
class ChunkGenerator:

    def __init__(self, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )

    def chunk(self, pages_df):

        records = []

        for _, row in pages_df.iterrows():

            chunks = self.splitter.split_text(row["text"])

            for chunk in chunks:

                records.append({"page_number": row["page_number"], "chunk_text": chunk})

        return pd.DataFrame(records)

In [5]:
class EmbeddingEngine:

    def __init__(self, model_name=EMBEDDING_MODEL):

        self.model = SentenceTransformer(model_name)

    def encode_documents(self, texts):

        vectors = self.model.encode(texts, convert_to_numpy=True)

        vectors = np.array(vectors).astype("float32")

        faiss.normalize_L2(vectors)

        return vectors

    def encode_query(self, query):

        vector = self.model.encode(query)

        vector = np.array([vector]).astype("float32")

        faiss.normalize_L2(vector)

        return vector

In [6]:
class FAISSRetriever:

    def __init__(self, embedding_dimension):

        self.index = faiss.IndexFlatIP(embedding_dimension)

    def add_vectors(self, vectors):

        self.index.add(vectors)

    def search(self, query_vector, top_k=TOP_K):

        scores, ids = self.index.search(query_vector, top_k)

        return scores, ids

In [7]:
class LocalLLM:

    def __init__(self, model_name=LLM_MODEL):

        tokenizer = AutoTokenizer.from_pretrained(model_name)

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True,
        )

        self.generator = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.1,
            return_full_text=False,
        )

    def generate(self, prompt):

        response = self.generator(prompt)

        return response[0]["generated_text"].strip()

In [8]:
class ConversationMemory:

    def __init__(self, window_size=6):

        self.history = []
        self.window_size = window_size

    def add(self, role, content):

        self.history.append({"role": role, "content": content})

    def get_history(self):

        return self.history[-self.window_size :]

    def format_history(self):

        output = ""

        for turn in self.get_history():

            output += f"{turn['role']}: " f"{turn['content']}\n"

        return output

In [9]:
class PDFChatbot:

    def __init__(self):

        print("Loading models...")

        self.parser = PDFParser()

        self.chunker = ChunkGenerator()

        self.embedder = EmbeddingEngine()

        self.llm = LocalLLM()

        self.memory = ConversationMemory()

        self.chunk_df = None
        self.retriever = None

        print("System Ready!")

In [10]:
def build_index(self, pdf_path):

    pages = self.parser.parse(pdf_path)

    self.chunk_df = self.chunker.chunk(pages)

    vectors = self.embedder.encode_documents(self.chunk_df["chunk_text"].tolist())

    self.retriever = FAISSRetriever(vectors.shape[1])

    self.retriever.add_vectors(vectors)


PDFChatbot.build_index = build_index

In [11]:
def retrieve(self, question):

    query_vector = self.embedder.encode_query(question)

    scores, ids = self.retriever.search(query_vector)

    results = self.chunk_df.iloc[ids[0]].copy()

    results["similarity"] = scores[0]

    return results


PDFChatbot.retrieve = retrieve

In [12]:
def build_prompt(self, question, retrieved_docs):

    context = "\n\n".join(retrieved_docs["chunk_text"].tolist())

    history = self.memory.format_history()

    prompt = f"""
You are a helpful PDF assistant.

Answer ONLY using the retrieved context.

Conversation History:
{history}

Retrieved Context:
{context}

Question:
{question}

If the answer cannot be found,
say:
"The information was not found in the document."

Answer:
"""

    return prompt


PDFChatbot.build_prompt = build_prompt

In [13]:
def ask(self, question):

    retrieved = self.retrieve(question)

    prompt = self.build_prompt(question, retrieved)

    answer = self.llm.generate(prompt)

    self.memory.add("user", question)

    self.memory.add("assistant", answer)

    return {"question": question, "answer": answer, "retrieved_docs": retrieved}


PDFChatbot.ask = ask

In [14]:
chatbot = PDFChatbot()

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 4fa43e10-482a-4a1f-a5f9-5d73b3063122)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].


Loading models...


'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: cd755656-10ef-49a9-9489-481901db5ab6)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 1ed29954-ac3c-4aa3-81cf-a321ae917061)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 50036fdf-2c5f-47b4-a041-e7a58a5a1080)')' thrown while

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device device because they were offloaded to the cpu and disk.


System Ready!


In [16]:
PDF_PATH = "data/uploads/annual_report.pdf"

chatbot.build_index(PDF_PATH)

print("PDF indexed successfully!")

PDF indexed successfully!


In [17]:
response = chatbot.ask("What was the company's revenue?")

print(response["answer"])

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
You are not running the flash-attention implementation, expect numerical differences.


The company's revenue for the year ended was 6,536.15.


Conversation History:


Retrieved Context:
Less : Closing Stock
Revenue as per contracted price 6,536.15 5,974.49 Finished goods (119.16) (108.21)
Add/(Less) : Adjustments Stock-in-trade (543.00) (375.94)
- Sales Return (154.11) (168.31) Work-in-progress (20.43) (17.76)
- Discounts (36.64) (26.35) (180.68) 7.73
Net revenue from sale of products and rendering of services 6,345.40 5,779.83
28 EMPLOYEE BENEFITS EXPENSE
Information about the Company’s performance obligations are summarized below :
For


In [19]:
# response = chatbot.ask("Did it increase compared to last year?")

# print(response["answer"])

In [20]:
# response["retrieved_docs"][["page_number", "similarity", "chunk_text"]]